<a href="https://colab.research.google.com/github/frasercrichton/ai-dde-hackthon/blob/feature%2Fleiden-guidelines-doc/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install  chromadb
!pip install git+https://github.com/huggingface/transformers.git triton

import logging


# Remove existing handlers (prevents duplicate logs)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.2 MB/s eta 0:00:0

In [36]:
from transformers import AutoTokenizer, AutoModel
from langchain.text_splitter import RecursiveCharacterTextSplitter
import torch

class EmbeddingsProcessor:

    model_name = 'sentence-transformers/all-MiniLM-L6-v2'

    def __init__(self):

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)
        self.model.eval()


    def create_embeddings(self, text):

        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True
        )
        logger.info(f'inputs: {inputs}')

        if torch.cuda.is_available():
            logger.info('cuda available')
            self.model.to('cuda')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
        else:
            logger.warning('cuda not available!')


        with torch.no_grad():
            outputs = self.model(**inputs)

                #             return {
                #     'text': text,
                #     'embeddings': embeddings,
                #     'metadata': {}
                # }


        return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().tolist()


In [37]:
embeddings_processor = EmbeddingsProcessor()

result = embeddings_processor.create_embeddings("my text")

print(result)

INFO: inputs: {'input_ids': tensor([[ 101, 2026, 3793,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


[0.15766668319702148, 0.1840507984161377, 0.032820120453834534, 0.02459709718823433, -0.06450758874416351, 0.06855998933315277, 1.2012853622436523, 0.5004308223724365, 0.8078906536102295, -0.03352796286344528, 0.176326185464859, 0.11282804608345032, 0.28723037242889404, -0.22774451971054077, 0.06691016256809235, -0.04883488267660141, -0.0014348700642585754, -0.4579885005950928, -0.9653513431549072, -0.26710349321365356, -0.4594149589538574, 0.9067121744155884, 0.13177165389060974, -0.08344840258359909, -0.3901810944080353, 0.46912652254104614, -0.6236799359321594, 0.6758586764335632, 0.0845165103673935, -0.40141159296035767, -0.6081819534301758, 0.0573454312980175, 0.9400818943977356, 0.2581678628921509, 0.01999201998114586, 0.03802858665585518, -0.4200325608253479, 0.09845513850450516, 0.2649827003479004, -0.036833446472883224, -0.17233234643936157, -0.8928815126419067, 0.47385552525520325, 0.1974477916955948, 0.5539321899414062, -0.15575262904167175, 0.13566988706588745, 0.1694144755

In [45]:
import chromadb


class RAGDatabase:

    def __init__(self, collection_name):
        self.vector_db = chromadb.Client()
        self.collection = self.vector_db.get_or_create_collection(name=collection_name)

    # delete

    def store_document(self, doc_id: str, document: str, embedding: list, metadata: dict = None):

        if metadata is None or not metadata:
            metadata = None

        if not isinstance(embedding, list):
            raise TypeError("Embedding must be a list of floats.")

        kwargs = {
                      "documents": [document],
                      "embeddings": [embedding],
                      "ids": [doc_id]
                  }

        if metadata is not None:
            kwargs["metadatas"] = [metadata]

        try:
            self.collection.add(**kwargs)
        except Exception as e:
            logging.error(f"Error storing document {doc_id}: {e}")

    def find_relevant_documents(
        self,
        query: str,
        headers: list = [],
        subheaders: list = [],
        context: list = [],
        n_results=3,
    ):
        """Find relevant documents for a given query."""

        # print(
        #     f'find_relevant_documents query: {query} headers: {headers}, subheaders {subheaders}, context: {context}'
        # )

        filter_dict = {}

        if headers:
            filter_dict['header'] = {'$in': headers}

        if subheaders:
            filter_dict['subheader'] = {'$in': subheaders}

        if context:
            filter_dict['context'] = {'$in': context}

        # If filter_dict is not empty, transform it into an OR query correctly
        if filter_dict:
            filter_dict = {'$or': [{key: value} for key, value in filter_dict.items()]}
        else:
            filter_dict = None  # Keep None when no filters exist
        # print(f'***** filter: {filter_dict}')

        return self.collection_query(query, filter_dict, n_results)

    #   def collection_embedding_query(self, query, filter, n_results=3):

    # query_embedding = generate_embedding("satellite images")

    # results = self.collection.query(
    #     query_embeddings=[query_embedding], where=filter_dict, n_results=3
    # )

    def collection_query(self, query, filter, n_results=3):
        """Find relevant documents for a given query."""

        results = self.collection.query(
            query_texts=[query], where=filter, n_results=n_results
        )
        # print(f'collection_query : {results}')
        return [
            {
                'text': doc_text,
                'id': results['ids'][0][i],
                'metadata': results['metadatas'][0][i],
            }
            for i, doc_text in enumerate(results['documents'][0])
        ]


In [47]:
rag_database = RAGDatabase(collection_name='my_collection')

document = "my text"
embedding = embeddings_processor.create_embeddings(document )
rag_database.store_document(doc_id='test', document= document, embedding= embedding)


INFO: inputs: {'input_ids': tensor([[ 101, 2026, 3793,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
